# 04 - Circuit Analysis & Emergent Dynamics

Investigating:
- Active hub neurons in the Central Complex
- Path efficiency in 3D navigation
- Emergent trajectory geometries (direct vs exploratory paths)


In [ ]:
import sys, os
from pathlib import Path

# Add project root and research/ to sys.path
cwd = Path.cwd().resolve()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / 'research' / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
    if (p / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from src.config import CONFIG
    from src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from src.environment import FlyNavigationEnv
    from src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
except ModuleNotFoundError:
    from research.src.config import CONFIG
    from research.src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from research.src.environment import FlyNavigationEnv
    from research.src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from research.src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch


## 1. Profiling Neural Activations in Navigation


In [ ]:
data = ConnectomeData()
neurons_df = data.fetch_all_neurons()
conn_df = data.fetch_connectivity(min_synapses=CONFIG['synapse_threshold'])
graph = data.build_networkx_graph(conn_df)

analysis = ConnectomeAnalysis(graph, neurons_df)
cx_graph = analysis.get_subgraph_by_roi('CX')

agent = ConnectomeConstrainedAgent(cx_graph, CONFIG['sensory_dim'], CONFIG['action_dim'], CONFIG)

# Load pre-trained weights if checkpoint exists
ck_paths = [
    Path('experiments/connectome_constrained/final_agent.pt'),
    Path('../experiments/connectome_constrained/final_agent.pt'),
    Path('../../research/experiments/connectome_constrained/final_agent.pt')
]
ck_file = next((p for p in ck_paths if p.exists()), None)
if ck_file:
    agent.load_state_dict(torch.load(ck_file, map_location='cpu'))
    print(f"Loaded trained checkpoint from {ck_file}")
else:
    print("No checkpoint found; running with initialized weights.")

env = FlyNavigationEnv(CONFIG)
act_probs, efficiencies = analyze_neuron_activations(agent, env, num_episodes=10)

print(f"Mean Path Efficiency: {np.mean(efficiencies):.3f} ± {np.std(efficiencies):.3f}")


## 2. Central Complex Hub Neuron Activation Distribution


In [ ]:
plot_activation_distribution(act_probs, neuron_ids=agent.neuron_ids, top_k=15)


## 3. 3D Trajectory Visualization


In [ ]:
obs = env.reset(seed=42)
hidden_state = None
done = False

agent.eval()
while not done:
    obs_t = torch.from_numpy(obs).unsqueeze(0).float()
    with torch.no_grad():
        logits, _, hidden_state = agent(obs_t, hidden_state)
        action = int(logits.argmax(dim=-1).item())
    obs, _, done, _ = env.step(action)

positions = np.array(env.trajectory_positions)

fig = plt.figure(figsize=(9, 6), dpi=120)
ax = fig.add_subplot(111, projection='3d')
ax.plot(positions[:, 0], positions[:, 1], positions[:, 2], color='#5B8CFF', linewidth=2.5, label='Agent Path')
ax.scatter(positions[0, 0], positions[0, 1], positions[0, 2], color='#3DDC97', s=100, label='Start')
ax.scatter(positions[-1, 0], positions[-1, 1], positions[-1, 2], color='#FF4D5E', s=100, label='Finish')
ax.scatter(env.goal[0], env.goal[1], env.goal[2], color='#FFC857', s=200, marker='*', label='Goal')

ax.set_title('Emergent 3D Navigation Trajectory', fontsize=14, fontweight='bold')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.legend()
plt.tight_layout()
plt.show()
